# Assignment Part 1 & 2 — Database Schema & ORM Setup

**System:** OR Scheduling System — Thai Government Hospital

This notebook demonstrates:
1. Python classes for all 10 conceptual entities (16 physical tables) using SQLAlchemy 2.0
2. Column definitions, Primary Keys, Foreign Keys
3. Building tables in the database with `Base.metadata.create_all()`
4. Verification via INFORMATION_SCHEMA queries

In [1]:
import sys
sys.path.insert(0, '../src')

from rich.console import Console
from rich.table import Table
from rich import print as rprint
console = Console()

## 1. ORM Model Classes

Each entity from the ER diagram maps to a SQLAlchemy `DeclarativeBase` class.

In [2]:
# Import all ORM model classes
from or_scheduler.models import (
    Base, Department, Staff, Room, Equipment, Patient,
    Case, Appointment,
    RoomReservation, StaffReservation, EquipmentReservation,
    RoomSchedule, StaffSchedule, EquipmentSchedule,
    Override, OverrideDisplacedAppointment, AuditLog
)

all_models = [
    Department, Staff, Room, Equipment, Patient,
    Case, Appointment,
    RoomReservation, StaffReservation, EquipmentReservation,
    RoomSchedule, StaffSchedule, EquipmentSchedule,
    Override, OverrideDisplacedAppointment, AuditLog
]

print(f"Loaded {len(all_models)} ORM model classes")

Loaded 16 ORM model classes


In [3]:
# Show column definitions for key tables
key_models = [Department, Staff, Room, Equipment, Patient, Case, Appointment, RoomReservation]

for model in key_models:
    table = model.__table__
    t = Table(title=f"{model.__name__} → {table.name}", show_header=True, header_style="bold cyan")
    t.add_column("Column", style="green")
    t.add_column("Type")
    t.add_column("Nullable")
    t.add_column("PK")
    t.add_column("FK")
    for col in table.columns:
        fk = ", ".join(str(f.target_fullname) for f in col.foreign_keys) if col.foreign_keys else ""
        t.add_row(
            col.name,
            str(col.type),
            str(col.nullable),
            "✓" if col.primary_key else "",
            fk
        )
    console.print(t)
    print()

              Department → departments               
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━┓
┃ Column        ┃ Type         ┃ Nullable ┃ PK ┃ FK ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━┩
│ department_id │ UUID         │ False    │ ✓  │    │
│ name          │ VARCHAR(200) │ False    │    │    │
│ building      │ VARCHAR(100) │ True     │    │    │
│ floor         │ INTEGER      │ True     │    │    │
│ created_at    │ TIMESTAMP    │ False    │    │    │
│ updated_at    │ TIMESTAMP    │ False    │    │    │
└───────────────┴──────────────┴──────────┴────┴────┘

                                Staff → staff                                
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column         ┃ Type         ┃ Nullable ┃ PK ┃ FK                        ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ staff_id       │ UUID         │ False    │ ✓  │                           │
│ name           │ VARCHAR(200) │ False    │    │                           │
│ role           │ VARCHAR(30)  │ False    │    │                           │
│ department_id  │ UUID         │ False    │    │ departments.department_id │
│ license_number │ VARCHAR(50)  │ True     │    │                           │
│ is_active      │ BOOLEAN      │ False    │    │                           │
│ created_at     │ TIMESTAMP    │ False    │    │                           │
│ updated_at     │ TIMESTAMP    │ False    │    │                           │
└────────────────┴──────────────┴──────────┴────┴───────────────────────────┘

                                Room → rooms                                 
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column          ┃ Type        ┃ Nullable ┃ PK ┃ FK                        ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ room_id         │ UUID        │ False    │ ✓  │                           │
│ room_code       │ VARCHAR(20) │ False    │    │                           │
│ room_type       │ VARCHAR(20) │ False    │    │                           │
│ department_id   │ UUID        │ True     │    │ departments.department_id │
│ is_laminar_flow │ BOOLEAN     │ False    │    │                           │
│ is_active       │ BOOLEAN     │ False    │    │                           │
│ created_at      │ TIMESTAMP   │ False    │    │                           │
│ updated_at      │ TIMESTAMP   │ False    │    │                           │
└─────────────────┴─────────────┴──────────┴────┴───────────────────────────┘

                      Equipment → equipment                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━┓
┃ Column                     ┃ Type         ┃ Nullable ┃ PK ┃ FK ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━┩
│ equipment_id               │ UUID         │ False    │ ✓  │    │
│ serial_number              │ VARCHAR(100) │ False    │    │    │
│ equipment_type             │ VARCHAR(100) │ False    │    │    │
│ status                     │ VARCHAR(20)  │ False    │    │    │
│ sterilization_duration_min │ INTEGER      │ False    │    │    │
│ last_sterilization_end     │ TIMESTAMP    │ True     │    │    │
│ created_at                 │ TIMESTAMP    │ False    │    │    │
│ updated_at                 │ TIMESTAMP    │ False    │    │    │
└────────────────────────────┴──────────────┴──────────┴────┴────┘

                Patient → patients                
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━┓
┃ Column     ┃ Type         ┃ Nullable ┃ PK ┃ FK ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━┩
│ patient_id │ UUID         │ False    │ ✓  │    │
│ hn         │ VARCHAR(20)  │ False    │    │    │
│ hosxp_ref  │ VARCHAR(50)  │ True     │    │    │
│ name       │ VARCHAR(200) │ False    │    │    │
│ age        │ INTEGER      │ True     │    │    │
│ blood_type │ VARCHAR(5)   │ True     │    │    │
│ allergies  │ TEXT         │ True     │    │    │
│ created_at │ TIMESTAMP    │ False    │    │    │
│ updated_at │ TIMESTAMP    │ False    │    │    │
└────────────┴──────────────┴──────────┴────┴────┘

                                      Case → cases                                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column                     ┃ Type         ┃ Nullable ┃ PK ┃ FK                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ case_id                    │ UUID         │ False    │ ✓  │                           │
│ patient_id                 │ UUID         │ False    │    │ patients.patient_id       │
│ department_id              │ UUID         │ False    │    │ departments.department_id │
│ initiated_by               │ UUID         │ False    │    │ staff.staff_id            │
│ procedure_type             │ VARCHAR(200) │ False    │    │                           │
│ urgency                    │ VARCHAR(20)  │ False    │    │                           │
│ status                     │ VARCHAR(20)  │ False    │    │                           │
│ clinical_notes             │ TEXT         │ True     │    │                           │
│ estimated_duration_minutes │ INTEGER      │ True     │    │                           │
│ created_at                 │ TIMESTAMP    │ False    │    │                           │
│ updated_at                 │ TIMESTAMP    │ False    │    │                           │
└────────────────────────────┴──────────────┴──────────┴────┴───────────────────────────┘

                   Appointment → appointments                    
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━┓
┃ Column         ┃ Type        ┃ Nullable ┃ PK ┃ FK             ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━┩
│ appointment_id │ UUID        │ False    │ ✓  │                │
│ case_id        │ UUID        │ False    │    │ cases.case_id  │
│ scheduled_date │ DATE        │ False    │    │                │
│ start_time     │ TIME        │ False    │    │                │
│ end_time       │ TIME        │ False    │    │                │
│ status         │ VARCHAR(20) │ False    │    │                │
│ version        │ INTEGER     │ False    │    │                │
│ confirmed_by   │ UUID        │ True     │    │ staff.staff_id │
│ confirmed_at   │ TIMESTAMP   │ True     │    │                │
│ created_at     │ TIMESTAMP   │ False    │    │                │
│ updated_at     │ TIMESTAMP   │ False    │    │                │
└────────────────┴─────────────┴──────────┴────┴────────────────┘

                       RoomReservation → room_reservations                       
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Column            ┃ Type        ┃ Nullable ┃ PK ┃ FK                          ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ reservation_id    │ UUID        │ False    │ ✓  │                             │
│ appointment_id    │ UUID        │ False    │    │ appointments.appointment_id │
│ room_id           │ UUID        │ False    │    │ rooms.room_id               │
│ status            │ VARCHAR(20) │ False    │    │                             │
│ reservation_start │ TIMESTAMP   │ False    │    │                             │
│ reservation_end   │ TIMESTAMP   │ False    │    │                             │
│ locked_at         │ TIMESTAMP   │ False    │    │                             │
│ created_at        │ TIMESTAMP   │ False    │    │                             │
│ updated_at        │ TIMESTAMP   │ False    │    │                             │
└───────────────────┴─────────────┴──────────┴────┴─────────────────────────────┘

## 2. Create Tables in Database

`Base.metadata.create_all(engine)` introspects all models and issues `CREATE TABLE` DDL.

In [4]:
from or_scheduler.database import engine
from sqlalchemy import text

# Ensure extensions exist
with engine.connect() as conn:
    conn.execute(text('CREATE EXTENSION IF NOT EXISTS "uuid-ossp";'))
    conn.execute(text('CREATE EXTENSION IF NOT EXISTS btree_gist;'))
    conn.commit()

# Create all tables
Base.metadata.create_all(engine)
print(f"✅ create_all() completed — {len(Base.metadata.tables)} tables registered in metadata")

✅ create_all() completed — 16 tables registered in metadata


## 3. Verify Tables Exist in PostgreSQL

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name;
    """))
    tables = [row[0] for row in result]

t = Table(title="Tables in PostgreSQL (public schema)", show_header=True, header_style="bold blue")
t.add_column("#", style="dim")
t.add_column("Table Name", style="green")
for i, name in enumerate(tables, 1):
    t.add_row(str(i), name)
console.print(t)
print(f"\nTotal: {len(tables)} tables")

  Tables in PostgreSQL (public schema)  
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #  ┃ Table Name                      ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1  │ appointments                    │
│ 2  │ audit_log                       │
│ 3  │ cases                           │
│ 4  │ departments                     │
│ 5  │ equipment                       │
│ 6  │ equipment_reservations          │
│ 7  │ equipment_schedules             │
│ 8  │ override_displaced_appointments │
│ 9  │ overrides                       │
│ 10 │ patients                        │
│ 11 │ room_reservations               │
│ 12 │ room_schedules                  │
│ 13 │ rooms                           │
│ 14 │ staff                           │
│ 15 │ staff_reservations              │
│ 16 │ staff_schedules                 │
└────┴─────────────────────────────────┘


Total: 16 tables


## 4. Verify GIST Exclusion Constraint (Anti-Double-Booking)

In [6]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT conname, contype, pg_get_constraintdef(oid) AS definition
        FROM pg_constraint
        WHERE conrelid = 'room_reservations'::regclass
        ORDER BY contype;
    """))
    constraints = result.fetchall()

t = Table(title="Constraints on room_reservations", show_header=True, header_style="bold magenta")
t.add_column("Name", style="yellow")
t.add_column("Type")
t.add_column("Definition", style="cyan")
for row in constraints:
    t.add_row(row[0], row[1], row[2])
console.print(t)

                                         Constraints on room_reservations                                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                                  ┃ Type ┃ Definition                                                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ck_room_res_status                    │ c    │ CHECK (((status)::text = ANY ((ARRAY['HELD'::character varying,  │
│                                       │      │ 'CONFIRMED'::character varying, 'RELEASED'::character varying,   │
│                                       │      │ 'COMPLETED'::character varying])::text[])))                      │
│ room_reservations_appointment_id_fkey │ f    │ FOREIGN KEY (appointment_id) REFERENCES                          │
│                                       │      │ appointments(appointment_id)                                     │
│ room_reservations_room_id_fkey        │ f    │ FOREIGN KEY (room_id) REFERENCES rooms(room_id)                  │
│ room_reservations_pkey                │ p    │ PRIMARY KEY (reservation_id)                                     │
│ uq_room_res_appt_room                 │ u    │ UNIQUE (appointment_id, room_id)                                 │
│ no_room_overlap                       │ x    │ EXCLUDE USING gist (room_id WITH =, tstzrange(reservation_start, │
│                                       │      │ reservation_end, '[)'::text) WITH &&) WHERE (((status)::text <>  │
│                                       │      │ ALL ((ARRAY['RELEASED'::character varying,                       │
│                                       │      │ 'COMPLETED'::character varying])::text[])))                      │
└───────────────────────────────────────┴──────┴──────────────────────────────────────────────────────────────────┘

## 5. Entity-Relationship Summary

In [7]:
t = Table(title="Entity Relationships", show_header=True, header_style="bold")
t.add_column("From Entity", style="green")
t.add_column("Relationship")
t.add_column("To Entity", style="blue")
t.add_column("Cardinality")

relationships = [
    ("Department", "employs", "Staff", "1:N"),
    ("Department", "manages", "Room", "1:N"),
    ("Department", "requests", "Case", "1:N"),
    ("Patient", "undergoes", "Case", "1:N"),
    ("Staff (Surgeon)", "initiates", "Case", "N:1"),
    ("Case", "requires", "Appointment", "1:N"),
    ("Appointment", "creates", "RoomReservation", "1:1"),
    ("Appointment", "creates", "StaffReservation", "1:N"),
    ("Appointment", "creates", "EquipmentReservation", "1:N"),
    ("RoomReservation", "reserves", "Room", "N:1"),
    ("StaffReservation", "reserves", "Staff", "N:1"),
    ("EquipmentReservation", "reserves", "Equipment", "N:1"),
    ("Schedule", "governs", "Room/Staff/Equipment", "N:1"),
    ("Appointment", "triggers", "Override", "1:0..1"),
    ("Override", "displaces", "Appointment", "1:1..N"),
]

for r in relationships:
    t.add_row(*r)
console.print(t)

                            Entity Relationships                            
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ From Entity          ┃ Relationship ┃ To Entity            ┃ Cardinality ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Department           │ employs      │ Staff                │ 1:N         │
│ Department           │ manages      │ Room                 │ 1:N         │
│ Department           │ requests     │ Case                 │ 1:N         │
│ Patient              │ undergoes    │ Case                 │ 1:N         │
│ Staff (Surgeon)      │ initiates    │ Case                 │ N:1         │
│ Case                 │ requires     │ Appointment          │ 1:N         │
│ Appointment          │ creates      │ RoomReservation      │ 1:1         │
│ Appointment          │ creates      │ StaffReservation     │ 1:N         │
│ Appointment          │ creates      │ EquipmentReservation │ 1:N         │
│ RoomReservation      │ reserves     │ Room                 │ N:1         │
│ StaffReservation     │ reserves     │ Staff                │ N:1         │
│ EquipmentReservation │ reserves     │ Equipment            │ N:1         │
│ Schedule             │ governs      │ Room/Staff/Equipment │ N:1         │
│ Appointment          │ triggers     │ Override             │ 1:0..1      │
│ Override             │ displaces    │ Appointment          │ 1:1..N      │
└──────────────────────┴──────────────┴──────────────────────┴─────────────┘

In [8]:
print("\n✅ Schema & ORM demonstration complete.")
print(f"   {len(Base.metadata.tables)} tables defined and verified in PostgreSQL.")
print("   GIST exclusion constraint confirms database-level double-booking prevention.")


✅ Schema & ORM demonstration complete.
   16 tables defined and verified in PostgreSQL.
   GIST exclusion constraint confirms database-level double-booking prevention.
